# 💰 Cost-Optimized Customer Churn Decision System

## Business Goal
Telecom companies lose significant revenue when customers churn. 
Retention campaigns cost money, so contacting every customer is inefficient.

This project builds a machine learning decision system that minimizes total business loss by optimizing prediction thresholds based on financial cost.

## 📌 Business Assumptions

To simulate real-world decision making:

- Cost of contacting a customer (retention offer): ₹500
- Revenue loss if a customer churns: ₹5000

Objective:
Minimize total business loss = (False Positives × ₹500) + (False Negatives × ₹5000)

In [13]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as imbPipeline
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,recall_score,precision_score,precision_recall_curve,average_precision_score,roc_auc_score,roc_curve,confusion_matrix
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance

In [14]:
df=pd.read_csv(r"C:\Users\pesak\OneDrive\ドキュメント\Pictures\Pictures\raja\aiml\project2-churn\data\WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"]=pd.to_numeric(df["TotalCharges"],errors="coerce")
print(df.head())
df=df.dropna()
df=df.drop("customerID",axis=1)
print(df["Churn"].value_counts(normalize=True))
print(df.isnull().sum())
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

## 📊 Dataset Overview

- Total customers: 7032 (after cleaning)
- Features: 20
- Target variable: Churn (27% positive rate)

Class imbalance exists, so evaluation metrics beyond accuracy are required.

In [15]:
X=df.drop("Churn",axis=1)
Y=df["Churn"]

x_train,x_temp,y_train,y_temp=train_test_split(X,Y,test_size=0.3,random_state=18,stratify=Y)
x_val,x_test,y_val,y_test=train_test_split(x_temp,y_temp,test_size=0.5,random_state=18,stratify=y_temp)

num_col=["SeniorCitizen","tenure","MonthlyCharges","TotalCharges"]
ch_col=["gender","Partner","Dependents","PhoneService","MultipleLines","InternetService","OnlineSecurity","OnlineBackup","DeviceProtection",
        "TechSupport","StreamingTV","StreamingMovies","Contract","PaperlessBilling","PaymentMethod"]

num_pipe=Pipeline([
    ("scaler",StandardScaler())
])
col_pipe=Pipeline([
    ("encode",OneHotEncoder(handle_unknown="ignore"))
])
preprocessor=ColumnTransformer([
    ("num_col",num_pipe,num_col),
    ("chr_col",col_pipe,ch_col)
])

## 🧱 Baseline Model

A DummyClassifier predicting the majority class was used as baseline.

This demonstrates why accuracy alone is misleading in imbalanced datasets.

In [16]:
dum=DummyClassifier(strategy="most_frequent")
dum.fit(x_train,y_train)
dum_prob=dum.predict_proba(x_test)[:,1]
dum_pred=dum.predict(x_test)
base_recall=recall_score(y_test,dum_pred)
base_precision=precision_score(y_test,dum_pred,zero_division=0)
base_accuracy=accuracy_score(y_test,dum_pred)
base_roc=roc_auc_score(y_test,dum_prob)
base_prc=average_precision_score(y_test,dum_prob)
print("BASE RESULTS:")
print("RECALL:",base_recall)
print("PRECISION:",base_precision)
print("ACCURACY:",base_accuracy)
print("ROC:",base_roc)
print("PRC:",base_prc)

BASE RESULTS:
RECALL: 0.0
PRECISION: 0.0
ACCURACY: 0.7345971563981043
ROC: 0.5
PRC: 0.26540284360189575


## 🤖 Model Development

Models evaluated:

- Logistic Regression (class_weight balanced)
- Random Forest (class_weight balanced)
- XGBoost
- SMOTE-based oversampling variants

Evaluation Metrics:
- ROC-AUC
- PR-AUC (important for imbalance)
- Recall & Precision

In [17]:
lg_pipe=Pipeline([
    ("preprocessor",preprocessor),
    ("model",LogisticRegression(class_weight="balanced",max_iter=1000,random_state=18,solver="liblinear"))
])
lg_param={
    "model__C":[0.1,1,10],
    "model__penalty":["l1","l2"]
}
lg_grid=GridSearchCV(
    lg_pipe,
    param_grid=lg_param,
    cv=3,
    scoring="roc_auc",
    n_jobs=2
)
lg_grid.fit(x_train,y_train)
lg_prob_val=lg_grid.predict_proba(x_val)[:,1]
precision,recall,threshold=precision_recall_curve(y_val,lg_prob_val)
lg_res=[]
for pr,re,th in zip(precision[:-1],recall[:-1],threshold):
    f1=(2*(pr*re)/(pr+re)) if(pr+re)!=0 else 0
    lg_res.append((f1,pr,re,th))
thres=max(lg_res,key=lambda x:x[0])
lg_thres=thres[3]
lg_prob=lg_grid.predict_proba(x_test)[:,1]
lg_pred=(lg_prob>lg_thres).astype(int)
lg_recall=recall_score(y_test,lg_pred)
lg_precision=precision_score(y_test,lg_pred)
lg_accuracy=accuracy_score(y_test,lg_pred)
lg_roc=roc_auc_score(y_test,lg_prob)
lg_prc=average_precision_score(y_test,lg_prob)
print("LG RESULTS:")
print("RECALL:",lg_recall)
print("PRECISION:",lg_precision)
print("ACCURACY:",lg_accuracy)
print("ROC:",lg_roc)
print("PRC:",lg_prc)
print("BEST_THRES:",lg_thres)
lg_imp=permutation_importance(
    lg_grid.best_estimator_,
    x_test,
    y_test,
    random_state=18,
    n_repeats=5,
    scoring="average_precision",
    n_jobs=2
)
lg_ftr_imp=pd.DataFrame({
    "Features":x_test.columns,
    "Importance":lg_imp.importances_mean
}).sort_values(by='Importance',ascending=False)
print(lg_ftr_imp.head(10))

LG RESULTS:
RECALL: 0.6607142857142857
PRECISION: 0.6085526315789473
ACCURACY: 0.7971563981042654
ROC: 0.8553271889400922
PRC: 0.6548592375843141
BEST_THRES: 0.6752557652688618
           Features  Importance
4            tenure    0.259921
7   InternetService    0.173848
17   MonthlyCharges    0.090656
18     TotalCharges    0.082848
14         Contract    0.040917
13  StreamingMovies    0.033604
12      StreamingTV    0.031564
6     MultipleLines    0.021437
11      TechSupport    0.011011
8    OnlineSecurity    0.008784


In [18]:
rf_pipe=Pipeline([
    ("preprocessor",preprocessor),
    ("model",RandomForestClassifier(bootstrap=True,max_features='sqrt',random_state=18,class_weight="balanced"))
])
rf_param={
    "model__n_estimators":[50,100],
    "model__max_depth":[3,5],
}
rf_grid=GridSearchCV(
    rf_pipe,
    param_grid=rf_param,
    cv=3,
    scoring="roc_auc",
    n_jobs=2
)
rf_grid.fit(x_train,y_train)
rf_prob_val=rf_grid.predict_proba(x_val)[:,1]
precision,recall,threshold=precision_recall_curve(y_val,rf_prob_val)
rf_res=[]
for pr,re,th in zip(precision[:-1],recall[:-1],threshold):
    f1=(2*(pr*re)/(pr+re)) if(pr+re)!=0 else 0
    rf_res.append((f1,pr,re,th))
thres=max(rf_res,key=lambda x:x[0])
rf_thres=thres[3]
rf_prob=rf_grid.predict_proba(x_test)[:,1]
rf_pred=(rf_prob>rf_thres).astype(int)
rf_recall=recall_score(y_test,rf_pred)
rf_precision=precision_score(y_test,rf_pred)
rf_accuracy=accuracy_score(y_test,rf_pred)
rf_roc=roc_auc_score(y_test,rf_prob)
rf_prc=average_precision_score(y_test,rf_prob)
print("RF RESULTS:")
print("RECALL:",rf_recall)
print("PRECISION:",rf_precision)
print("ACCURACY:",rf_accuracy)
print("ROC:",rf_roc)
print("PRC:",rf_prc)
print("BEST_THRES:",rf_thres)
rf_imp=permutation_importance(
    rf_grid.best_estimator_,
    x_test,
    y_test,
    scoring="average_precision",
    n_repeats=5,
    n_jobs=2
)
rf_ftr_imp=pd.DataFrame({
    "Features":x_test.columns,
    "Importance":rf_imp.importances_mean
}).sort_values(by="Importance",ascending=False)
print(rf_ftr_imp.head(10))

RF RESULTS:
RECALL: 0.7142857142857143
PRECISION: 0.6042296072507553
ACCURACY: 0.8
ROC: 0.8538271889400921
PRC: 0.6507526968039925
BEST_THRES: 0.6213947701812375
           Features  Importance
14         Contract    0.094844
7   InternetService    0.062786
4            tenure    0.043565
8    OnlineSecurity    0.025103
11      TechSupport    0.017588
18     TotalCharges    0.009004
16    PaymentMethod    0.004282
12      StreamingTV    0.003838
13  StreamingMovies    0.003714
17   MonthlyCharges    0.002558


In [19]:
xgb_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        random_state=18,
        eval_metric="logloss",
        scale_pos_weight=(len(y_train[y_train==0]) / len(y_train[y_train==1]))
    ))
])
xgb_pipe.fit(x_train, y_train)
xgb_pred = xgb_pipe.predict(x_test)
xgb_prob = xgb_pipe.predict_proba(x_test)[:,1]
xgb_recall = recall_score(y_test, xgb_pred)
xgb_precision = precision_score(y_test, xgb_pred)
xgb_accuracy = accuracy_score(y_test, xgb_pred)
xgb_roc = roc_auc_score(y_test, xgb_prob)
xgb_prc = average_precision_score(y_test, xgb_prob)

print("XGBOOST RESULTS:")
print("Recall:", xgb_recall)
print("Precision:", xgb_precision)
print("Accuracy:", xgb_accuracy)
print("ROC:", xgb_roc)
print("PRC:", xgb_prc)

XGBOOST RESULTS:
Recall: 0.6892857142857143
Precision: 0.5331491712707183
Accuracy: 0.7573459715639811
ROC: 0.8228133640552995
PRC: 0.6342744657930874


In [20]:
smote=SMOTE(random_state=18)

lg_smt_pipe=imbPipeline([
    ("preprocessor",preprocessor),
    ("smote",smote),
    ("model",LogisticRegression(max_iter=1000,random_state=18,solver="liblinear"))
])
lg_smt_param={
    "model__C":[0.1,1,10],
    "model__penalty":["l1","l2"]
}
lg_smt_grid=GridSearchCV(
    lg_smt_pipe,
    param_grid=lg_smt_param,
    cv=3,
    scoring="roc_auc",
    n_jobs=2
)
lg_smt_grid.fit(x_train,y_train)
lg_smt_prob_val=lg_smt_grid.predict_proba(x_val)[:,1]
precision,recall,threshold=precision_recall_curve(y_val,lg_smt_prob_val)
lg_smt_res=[]
for pr,re,th in zip(precision[:-1],recall[:-1],threshold):
    f1=(2*(pr*re)/(pr+re)) if(pr+re)!=0 else 0
    lg_smt_res.append((f1,pr,re,th))
thres=max(lg_smt_res,key=lambda x:x[0])
lg_smt_thres=thres[3]
lg_smt_prob=lg_smt_grid.predict_proba(x_test)[:,1]
lg_smt_pred=(lg_smt_prob>lg_smt_thres).astype(int)
lg_smt_recall=recall_score(y_test,lg_smt_pred)
lg_smt_precision=precision_score(y_test,lg_smt_pred)
lg_smt_accuracy=accuracy_score(y_test,lg_smt_pred)
lg_smt_roc=roc_auc_score(y_test,lg_smt_prob)
lg_smt_prc=average_precision_score(y_test,lg_smt_prob)
print("LG SMT RESULTS:")
print("RECALL:",lg_smt_recall)
print("PRECISION:",lg_smt_precision)
print("ACCURACY:",lg_smt_accuracy)
print("ROC:",lg_smt_roc)
print("PRC:",lg_smt_prc)
print("BEST_THRES:",lg_smt_thres)
lg_smt_imp=permutation_importance(
    lg_smt_grid.best_estimator_,
    x_test,
    y_test,
    random_state=18,
    n_repeats=5,
    scoring="average_precision",
    n_jobs=2
)
lg_smt_ftr_imp=pd.DataFrame({
    "Features":x_test.columns,
    "Importance":lg_smt_imp.importances_mean
}).sort_values(by='Importance',ascending=False)
print(lg_smt_ftr_imp.head(10))

LG SMT RESULTS:
RECALL: 0.8357142857142857
PRECISION: 0.5154185022026432
ACCURACY: 0.747867298578199
ROC: 0.8535069124423964
PRC: 0.6539543782509735
BEST_THRES: 0.48883047576846467
          Features  Importance
4           tenure    0.271720
17  MonthlyCharges    0.106030
18    TotalCharges    0.088748
14        Contract    0.036562
8   OnlineSecurity    0.015592
5     PhoneService    0.013785
11     TechSupport    0.013462
16   PaymentMethod    0.008608
1    SeniorCitizen    0.005160
6    MultipleLines    0.003170


In [21]:
rf_smt_pipe=imbPipeline([
    ("preprocessor",preprocessor),
    ("smote",smote),
    ("model",RandomForestClassifier(bootstrap=True,max_features='sqrt',random_state=18))
])
rf_smt_param={
    "model__n_estimators":[50,100],
    "model__max_depth":[3,5],
}
rf_smt_grid=GridSearchCV(
    rf_smt_pipe,
    param_grid=rf_smt_param,
    cv=3,
    scoring="roc_auc",
    n_jobs=2
)
rf_smt_grid.fit(x_train,y_train)
rf_smt_prob_val=rf_smt_grid.predict_proba(x_val)[:,1]
precision,recall,threshold=precision_recall_curve(y_val,rf_smt_prob_val)
rf_smt_res=[]
for pr,re,th in zip(precision[:-1],recall[:-1],threshold):
    f1=(2*(pr*re)/(pr+re)) if(pr+re)!=0 else 0
    rf_smt_res.append((f1,pr,re,th))
thres=max(rf_smt_res,key=lambda x:x[0])
rf_smt_thres=thres[3]
rf_smt_prob=rf_smt_grid.predict_proba(x_test)[:,1]
rf_smt_pred=(rf_smt_prob>rf_smt_thres).astype(int)
rf_smt_recall=recall_score(y_test,rf_smt_pred)
rf_smt_precision=precision_score(y_test,rf_smt_pred)
rf_smt_accuracy=accuracy_score(y_test,rf_smt_pred)
rf_smt_roc=roc_auc_score(y_test,rf_smt_prob)
rf_smt_prc=average_precision_score(y_test,rf_smt_prob)
print("RF SMT RESULTS:")
print("RECALL:",rf_smt_recall)
print("PRECISION:",rf_smt_precision)
print("ACCURACY:",rf_smt_accuracy)
print("ROC:",rf_smt_roc)
print("PRC:",rf_smt_prc)
print("BEST_THRES:",rf_smt_thres)
rf_smt_imp=permutation_importance(
    rf_smt_grid.best_estimator_,
    x_test,
    y_test,
    scoring="average_precision",
    n_repeats=5,
    n_jobs=2
)

rf_smt_ftr_imp=pd.DataFrame({
    "Features":x_test.columns,
    "Importance":rf_smt_imp.importances_mean
}).sort_values(by="Importance",ascending=False)
print(rf_smt_ftr_imp.head(10))

RF SMT RESULTS:
RECALL: 0.8178571428571428
PRECISION: 0.5276497695852534
ACCURACY: 0.7573459715639811
ROC: 0.8472142857142857
PRC: 0.6281860917478397
BEST_THRES: 0.4927638758858709
            Features  Importance
14          Contract    0.070202
11       TechSupport    0.042727
16     PaymentMethod    0.038944
8     OnlineSecurity    0.036790
7    InternetService    0.035127
4             tenure    0.024888
17    MonthlyCharges    0.013234
15  PaperlessBilling    0.006359
10  DeviceProtection    0.006101
12       StreamingTV    0.005036


In [22]:
results = pd.DataFrame([
    ["Dummy", base_roc, base_prc, base_recall, base_precision],
    ["Logistic Balanced", lg_roc, lg_prc, lg_recall, lg_precision],
    ["RF Balanced", rf_roc, rf_prc, rf_recall, rf_precision],
    ["XGB",xgb_roc,xgb_prc,xgb_recall,xgb_precision],
    ["Logistic SMOTE", lg_smt_roc, lg_smt_prc, lg_smt_recall, lg_smt_precision],
    ["RF SMOTE", rf_smt_roc, rf_smt_prc, rf_smt_recall, rf_smt_precision],
], columns=["Model", "ROC_AUC", "PR_AUC", "Recall", "Precision"])

print(results.sort_values(by="PR_AUC", ascending=False))


               Model   ROC_AUC    PR_AUC    Recall  Precision
1  Logistic Balanced  0.855327  0.654859  0.660714   0.608553
4     Logistic SMOTE  0.853507  0.653954  0.835714   0.515419
2        RF Balanced  0.853827  0.650753  0.714286   0.604230
3                XGB  0.822813  0.634274  0.689286   0.533149
5           RF SMOTE  0.847214  0.628186  0.817857   0.527650
0              Dummy  0.500000  0.265403  0.000000   0.000000


In [23]:
def compute_cost(y_true,y_pred,fn_cost=5000,fp_cost=500):
    tn,fp,fn,tp=confusion_matrix(y_true,y_pred).ravel()
    return fp*fp_cost + fn*fn_cost
rf_cost = compute_cost(y_test, rf_pred)
xgb_cost = compute_cost(y_test, xgb_pred)
lg_cost = compute_cost(y_test, lg_pred)
lg_smt_cost = compute_cost(y_test, lg_smt_pred)
rf_smt_cost = compute_cost(y_test, rf_smt_pred)
print("RF Cost:", rf_cost)
print("LG Cost:", lg_cost)
print("LG SMOTE Cost:", lg_smt_cost)
print("RF SMOTE Cost:", rf_smt_cost)
print("XGB Cost:",xgb_cost)

RF Cost: 465500
LG Cost: 534500
LG SMOTE Cost: 340000
RF SMOTE Cost: 357500
XGB Cost: 519500


## 💰 Cost-Based Threshold Optimization

Traditional metrics like F1-score do not directly reflect business impact.

Instead of selecting threshold based on F1, 
we optimized the classification threshold to minimize total business loss.

This significantly reduced simulated loss from ₹357,500 to ₹263,000.

In [24]:
rf_smt_prob_val = rf_smt_grid.predict_proba(x_val)[:,1]

def compute_cost_from_probs(y_true, probs, threshold, fn_cost=5000, fp_cost=500):
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    return fp * fp_cost + fn * fn_cost

thresholds = np.linspace(0.01, 0.99, 200)

costs = []
for t in thresholds:
    cost = compute_cost_from_probs(y_val, rf_smt_prob_val, t)
    costs.append(cost)

best_threshold = thresholds[np.argmin(costs)]
min_cost = min(costs)

print("Best Threshold (Cost Optimized):", best_threshold)
print("Minimum Validation Cost:", min_cost)

rf_smt_prob_test = rf_smt_grid.predict_proba(x_test)[:,1]
rf_smt_pred_cost = (rf_smt_prob_test >= best_threshold).astype(int)

final_cost = compute_cost(y_test, rf_smt_pred_cost)

print("Final Test Cost (Cost Optimized):", final_cost)

Best Threshold (Cost Optimized): 0.275929648241206
Minimum Validation Cost: 280500
Final Test Cost (Cost Optimized): 263000


## 🏁 Conclusion

Key Insights:

- Simpler models (Logistic Regression) performed competitively with boosting.
- SMOTE increased recall but required threshold adjustment.
- Cost-based threshold tuning reduced total business loss by ₹94,500.
- Business-aligned evaluation leads to better decision systems than metric-only optimization.

This project demonstrates how machine learning should align with financial objectives rather than just accuracy.